### Load ISM dataset and compute shift vectors

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_flim.tools_phasor as flim
import brighteyes_ism.analysis.APR_lib as apr
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.analysis.Tools_lib as tools
import brighteyes_ism.dataio.mcs as mcs
from tqdm import tqdm


In [ ]:
data = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38.h5", "r")
print(data.keys())
image = data["data"]
image_4D = image[0, 0,...]
print(image_4D.shape)
image_3d = np.sum(image_4D, axis = -2)
print(image_3d.shape)
shift_vectors, err = apr.ShiftVectors(image_3d, usf = 100, ref = 12)
gr.PlotShiftVectors(shift_vectors)
print (shift_vectors)     

### Do pixel reassignment for each bin of the image

In [ ]:
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_750_data-10-04-2024-17-42-38", 'w') as f:
     x_size, y_size, bin_size, channel_size = 750, 750, image_4D.shape[2], image_4D.shape[3]
# Create an empty dataset with dimensions (x,y,t, ch)
     dataset_shape = (x_size, y_size, bin_size, channel_size)
     h5_dataset = f.create_dataset('data', shape=dataset_shape, dtype=np.uint16)
    

     

     for bin in tqdm(range(image_4D.shape[-2])):
         h5_dataset[:, :, bin, :] = apr.Reassignment(shift_vectors, image_4D[1000:1750, 1200:1950, bin, :], mode = 'interp')
   



### Sum over channels to get the (x, y, t) image

In [ ]:
f = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_750_data-10-04-2024-17-42-38", 'r')
h5_dataset = f["data"]
print(h5_dataset.shape)
#h5_dataset_sum = np.sum(h5_dataset, axis=3)

In [ ]:
h5_dataset_sum = np.sum(h5_dataset, axis=-1)
#image_reassigned = apr.Reassignment(shift_vectors, image[:, :, 0, :])

In [ ]:
print(h5_dataset_sum.shape)

### Sum over the time bins to get the intensity (x, y) image

In [ ]:

data_histograms = np.sum(h5_dataset_sum, axis = -1)
print(data_histograms.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
plt.figure()
plt.hist(data_histograms.flatten(), bins = 100, range = (0, 500))

### compute the phasor on the (x, y, t) image for each pixel

In [ ]:
#Save the phasors in the image after bin per bin pixel reassignment
flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\APR_750_data-10-04-2024-17-42-38", data_input = h5_dataset_sum)

In [ ]:
%matplotlib widget

hf_phasors_per_pixel = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_750_data-10-04-2024-17-42-38_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel.keys())

phasors_pix = hf_phasors_per_pixel["h5_dataset_phasor_pix"]  # data with phasors in each pixel
#fasors_pix[1:100, 1:100]
print(phasors_pix.shape)

flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False)

### Extract phasor of the IRF for correcting the measured phasors

In [ ]:
data_path_irf = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-18-31-53.h5"
data_irf = h5py.File(data_path_irf)

image_irf = data_irf["data"]
data_hist_irf = np.sum(image_irf, axis=(0, 1, 2, 3))
print(data_hist_irf.shape)



In [ ]:
bins_irf = np.sum(data_hist_irf, axis = -1)
print(bins_irf.shape)
phasor_i = flim.calculate_phasor(bins_irf)
print(phasor_i)

### Extract the phasor of the laser and use it to shift the phasors of the data

In [ ]:
#load the histogram of the laser trigger signal (26th channel) acquired during the sample's acquisition
import brighteyes_ism.dataio.mcs as mcs
data_path = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38.h5"
data_extra, _ = mcs.load(data_path, key="data_channels_extra")
data_laser = data_extra[:, :, :, :, :, 1]
print(data_laser.shape)
data_laser_hist = np.sum(data_laser, axis = (0,1,2,3))
phasor_laser = flim.calculate_phasor(data_laser_hist)
print(phasor_laser)

In [ ]:
#load the histogram of the laser trigger signal (26th channel) acquired with the IRF acquisition
data_path_irf = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-18-31-53.h5"
data_extra_irf, _ = mcs.load(data_path_irf, key="data_channels_extra")
data_laser_irf = data_extra_irf[:, :, :, :, :, 1]
print(data_laser_irf.shape)
data_laser_hist_irf = np.sum(data_laser_irf, axis = (0,1,2,3))
phasor_laser_irf = flim.calculate_phasor(data_laser_hist_irf)
print(phasor_laser_irf)

In [ ]:
corr = flim.correction_phasor(data_laser_hist, data_laser_hist_irf)
print(corr)
phasor_corrected = phasors_pix * corr / phasor_i

### Display the phasor plot of the data realigned¶

In [ ]:
flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first')

### Lifetime analysis and FLISM display

In [ ]:
tau_phi = flim.calculate_tau_phi(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_phi.shape)


In [ ]:
tau_m = flim.calculate_tau_m(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_m.shape)

In [ ]:
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (-6, 6), bins = 50)

In [ ]:
tau_m_data = 1e9*tau_m.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 13), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms, tau_m*1e9, pxsize = 0.05, pxdwelltime = 324, lifetime_bounds = (1.3, 6), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\APR_ups100_convallaria_C_.pdf", dpi = 900)

### Plot intensities of the channels after APR

In [ ]:
f = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_750_data-10-04-2024-17-42-38", 'r')
h5_dataset = f["data"]
data_x_y_ch = np.sum(h5_dataset, axis = -2)
print(data_x_y_ch.shape)

In [ ]:
fig = gr.ShowDataset(data_x_y_ch, pxsize = 0.05)

### Plot intensity image (Nx x Ny)

In [ ]:
gr.ShowImg(data_histograms, pxsize_x = 0.05)